# Etapa 1 — Baselines: DummyClassifier e Regressão Logística

Treinamento de modelos baseline para previsão de churn em telecomunicações.  
Todos os experimentos são rastreados no **MLflow** com validação cruzada estratificada (5-fold).

**Modelos:** DummyClassifier (most_frequent, stratified) e Regressão Logística  
**Métricas monitoradas:** Accuracy, AUC-ROC, PR-AUC, F1, F1-weighted


In [ ]:
import logging
import sys
import warnings

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

sys.path.insert(0, '../src')
from preprocessing import RANDOM_STATE, build_preprocessor, load_and_clean_data, prepare_features

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
)
logger = logging.getLogger('baselines')
logger.info('Bibliotecas carregadas com sucesso.')

In [ ]:
CV_FOLDS = 5
EXPERIMENT_NAME = 'telco-churn-baselines'

SCORING = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'average_precision': 'average_precision',
    'f1': 'f1',
    'f1_weighted': 'f1_weighted',
}

logger.info(
    'Configuracao: RANDOM_STATE=%d, CV_FOLDS=%d, EXPERIMENT=%s',
    RANDOM_STATE, CV_FOLDS, EXPERIMENT_NAME,
)

In [ ]:
df = load_and_clean_data()
X, y = prepare_features(df)

logger.info(
    'Classe alvo - 0 (Sem Churn): %d (%.1f%%) | 1 (Churn): %d (%.1f%%)',
    (y == 0).sum(), 100 * (y == 0).mean(),
    (y == 1).sum(), 100 * (y == 1).mean(),
)
display(df.head(3))

In [ ]:
mlflow.set_experiment(EXPERIMENT_NAME)

dataset_info = mlflow.data.from_pandas(
    df,
    name='telco_customer_churn',
    targets='target',
)

logger.info('Experimento MLflow configurado: %s', EXPERIMENT_NAME)

In [ ]:
def run_cv_and_log(model_name, clf_pipeline, extra_params):
    """Execute cross-validation estratificada e registra resultados no MLflow."""
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    logger.info('Iniciando CV para: %s', model_name)

    cv_results = cross_validate(clf_pipeline, X, y, cv=cv, scoring=SCORING)

    metrics = {}
    for key, values in cv_results.items():
        if key.startswith('test_'):
            name = key[5:]
            metrics[name] = float(values.mean())
            metrics[name + '_std'] = float(values.std())

    params = {
        'model_name': model_name,
        'cv_folds': CV_FOLDS,
        'random_state': RANDOM_STATE,
        'dataset_rows': len(X),
        'dataset_features': X.shape[1],
    }
    params.update(extra_params)

    with mlflow.start_run(run_name=model_name) as run:
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.log_input(dataset_info, context='training')
        mlflow.sklearn.log_model(clf_pipeline, artifact_path='model')
        run_id = run.info.run_id

    logger.info(
        '%s | AUC-ROC=%.4f (+/-%.4f) | F1=%.4f | Accuracy=%.4f',
        model_name,
        metrics['roc_auc'], metrics['roc_auc_std'],
        metrics['f1'], metrics['accuracy'],
    )
    return {'run_id': run_id, 'metrics': metrics, 'model_name': model_name}

## DummyClassifier Baseline

Treinamos duas estratégias do DummyClassifier como **limite inferior de desempenho**:
- **most_frequent**: sempre prediz a classe majoritária (Não Churn — ~73%)
- **stratified**: prediz aleatoriamente respeitando a distribuição das classes


In [ ]:
results_list = []

for strategy in ['most_frequent', 'stratified']:
    dummy_clf = DummyClassifier(strategy=strategy, random_state=RANDOM_STATE)
    clf_pipeline = Pipeline(steps=[
        ('preprocessor', build_preprocessor()),
        ('classifier', dummy_clf),
    ])

    result = run_cv_and_log(
        model_name='DummyClassifier_' + strategy,
        clf_pipeline=clf_pipeline,
        extra_params={'strategy': strategy},
    )
    results_list.append(result)

logger.info('%d runs DummyClassifier registradas no MLflow.', len(results_list))

In [ ]:
summary_rows = []
for r in results_list:
    row = {'modelo': r['model_name']}
    for m in ['accuracy', 'roc_auc', 'average_precision', 'f1', 'f1_weighted']:
        row[m] = round(r['metrics'].get(m, 0.0), 4)
    summary_rows.append(row)

results_df = pd.DataFrame(summary_rows).set_index('modelo')
display(results_df)
logger.info('Resumo DummyClassifier exibido.')

## Regressão Logística Baseline

Treinamos Regressão Logística como baseline com capacidade de aprendizado real.  
Usamos `class_weight='balanced'` para lidar com o desbalanceamento (~73% sem churn vs ~27% com churn).

**Esperado:** superar significativamente o DummyClassifier, especialmente em AUC-ROC e F1.


In [ ]:
lr_params = {
    'C': 1.0,
    'solver': 'lbfgs',
    'max_iter': 1000,
    'class_weight': 'balanced',
}

lr_clf = LogisticRegression(random_state=RANDOM_STATE, **lr_params)
lr_pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', lr_clf),
])

result_lr = run_cv_and_log(
    model_name='LogisticRegression',
    clf_pipeline=lr_pipeline,
    extra_params=lr_params,
)
results_list.append(result_lr)

logger.info('Regressao Logistica concluida e registrada no MLflow.')

## Comparação de Modelos Baseline

Tabela comparativa com métricas de validação cruzada estratificada (média ± desvio padrão, 5-fold).


In [ ]:
comparison_rows = []
for r in results_list:
    row = {'Modelo': r['model_name']}
    for metric in ['accuracy', 'roc_auc', 'average_precision', 'f1', 'f1_weighted']:
        mean_val = r['metrics'].get(metric, 0.0)
        std_val = r['metrics'].get(metric + '_std', 0.0)
        row[metric] = str(round(mean_val, 4)) + ' +/- ' + str(round(std_val, 4))
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index('Modelo')
display(comparison_df)

logger.info('Tabela comparativa exibida. Todos os experimentos registrados no MLflow.')

## Conclusão

Todos os experimentos foram registrados no MLflow com:
- **Parâmetros**: hiperparâmetros do modelo, configuração de CV (folds, random_state)
- **Métricas**: accuracy, AUC-ROC, PR-AUC, F1 e F1-weighted (média e desvio padrão)
- **Dataset**: versão e informações do conjunto de dados (7.043 registros, 19 features)
- **Artefatos**: modelo serializado via MLflow sklearn

A Regressão Logística supera significativamente os DummyClassifiers,
estabelecendo o **baseline de referência** para comparação com a MLP PyTorch na Etapa 2.
